# 2026-03-07 TODO

|TODO|내용|비고|구현 일시|
|:---:|:---:|:---:|:---:|
|[ ]|1. GCP BigQuery에서 Polars으로 상담 데이터 클러스터링||2026/03/07-2026/03/08|

In [1]:
from google.cloud import bigquery
from google.oauth2 import service_account

key_path = '/workspaces/Psychological-counseling-researching/.key/testprojects-453622-d1f78fcce8b7.json'
credentials = service_account.Credentials.from_service_account_file(key_path)

client = bigquery.Client(credentials=credentials, project=credentials.project_id)
print(f"Client created with project: {client.project}")

Client created with project: testprojects-453622


## 1. GCP BigQuery에서 Polars으로 상담 데이터 클러스터링

|TODO|내용|비고|구현 일시|
|:---:|:---:|:---:|:---:|
|[ ]|텍스트 임베딩 처리|텍스트 임베딩을 하고 Google BigQuery에 저장.|2026/03/07-2026/03/08|

In [2]:
# Load model directly
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained("madatnlp/km-bert")
model = AutoModel.from_pretrained("madatnlp/km-bert")

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:06<00:00, 32.12it/s]
BertModel LOAD REPORT from: madatnlp/km-bert
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
sample_text = "오늘 선생님과 세 번째 상담 시작하도록 할게요."
sample_inputs = tokenizer(sample_text, return_tensors="pt", truncation=True, max_length=256)
display(sample_inputs)

{'input_ids': tensor([[   2, 1248, 6276,   24,  248,  781, 5266,  576, 1511,  334,  101,  239,
            5,    3]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [4]:
schema_query = """
SELECT column_name, data_type
FROM `testprojects-453622.Psychological_counseling_data.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name = 'morpheme_classification'
ORDER BY ordinal_position
"""

schema_df = client.query(schema_query).to_dataframe()
display(schema_df)

,column_name,data_type
0,file_code,STRING
1,speaker,STRING
2,session_no,INT64
3,timeline_index,INT64
4,end_content_index,INT64
5,split_row_index,INT64
6,split_contents,STRING


In [ ]:
import pandas as pd
import torch
from tqdm.auto import tqdm

source_table = "`testprojects-453622.Psychological_counseling_data.morpheme_classification`"
target_table = "testprojects-453622.Psychological_counseling_data.morpheme_classification_kmbert_embedding"

QUERY = f"""
SELECT
    file_code,
    speaker,
    session_no,
    timeline_index,
    end_content_index,
    split_row_index,
    split_contents
FROM {source_table}
WHERE split_contents IS NOT NULL
  AND TRIM(split_contents) != ''
ORDER BY
    end_content_index ASC,
    timeline_index ASC,
    split_row_index ASC
"""

rows_df = client.query(QUERY).to_dataframe()
print(f"rows: {len(rows_df):,}")

if rows_df.empty:
    raise ValueError("No rows found to embed.")

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()

def mean_pooling(last_hidden_state: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    # Masked mean pooling to ignore paddings.
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    masked = last_hidden_state * mask
    summed = masked.sum(dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts

texts = rows_df["split_contents"].astype(str).tolist()
batch_size = 32
embeddings: list[list[float]] = []

with torch.no_grad():
    for start in tqdm(range(0, len(texts), batch_size), desc="Embedding"):
        batch_texts = texts[start:start + batch_size]
        encoded = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=256,
        )
        encoded = {k: v.to(device) for k, v in encoded.items()}
        outputs = model(**encoded)
        pooled = mean_pooling(outputs.last_hidden_state, encoded["attention_mask"])
        embeddings.extend(pooled.cpu().tolist())

result_df = rows_df.copy()
result_df["embedding_model"] = "madatnlp/km-bert"
result_df["embedding_dim"] = len(embeddings[0])
result_df["content_embedding"] = embeddings
result_df["embedded_at"] = pd.Timestamp.utcnow()

schema = [
    bigquery.SchemaField("file_code", "STRING"),
    bigquery.SchemaField("speaker", "STRING"),
    bigquery.SchemaField("session_no", "INT64"),
    bigquery.SchemaField("timeline_index", "INT64"),
    bigquery.SchemaField("end_content_index", "INT64"),
    bigquery.SchemaField("split_row_index", "INT64"),
    bigquery.SchemaField("split_contents", "STRING"),
    bigquery.SchemaField("embedding_model", "STRING"),
    bigquery.SchemaField("embedding_dim", "INT64"),
    bigquery.SchemaField("content_embedding", "FLOAT64", mode="REPEATED"),
    bigquery.SchemaField("embedded_at", "TIMESTAMP"),
]

load_job_config = bigquery.LoadJobConfig(
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
    schema=schema,
)

load_job = client.load_table_from_dataframe(
    result_df,
    target_table,
    job_config=load_job_config,
    parquet_compression="SNAPPY",
)
load_job.result()

print(f"Saved {len(result_df):,} rows to {target_table}")
print(f"Embedding dim: {result_df['embedding_dim'].iloc[0]}")

rows: 119,005


Embedding:   4%|▍         | 165/3719 [05:10<1:48:59,  1.84s/it]